# FEATURES_FINAL Doğrulama

**Tarih:** 2026-05-11
**Amac:** 9 analizden cıkan tum feature'lari tek tabloya topla, V6 aday feature'larin leakage testini yap, multicollinearity (VIF) kontrol et, ML V6 icin final karar ver.

## Plan
1. Tum V6 EKLE feature'larini (9) topla
2. V6 aday feature'larini (4) hesapla + leakage testi
3. Pairwise correlation matrix (multicollinearity ilk gozlem)
4. VIF (Variance Inflation Factor) testi
5. Feature production pipeline ozet
6. Final karar tablosu + ML V6 oneri

## Cikti
- `features_final_master.csv` — KAPINO basina tum feature'lar
- `FEATURES_FINAL.md` icin hazır veriler


---
## 1. Tum Feature'lari Tek Tabloya Topla

Her arac icin master profil + 9 V6 EKLE feature.


In [1]:
# BOLUM 1: Feature toplama
import pandas as pd
import numpy as np
import json as _json
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# Ana veri
df = pd.read_csv("panel_data/temiz_veri/ariza_model.csv", low_memory=False)
df["OLAYTARIHI"] = pd.to_datetime(df["OLAYTARIHI"], format="mixed")
mask_b = df["YAKITTURU"]=="Bilinmiyor"
df.loc[mask_b & df["MODEL"].str.contains("CNG", na=False), "YAKITTURU"] = "CNG"
df.loc[mask_b & ~df["MODEL"].str.contains("CNG", na=False), "YAKITTURU"] = "MOTORIN"
df = df[df["YAKITTURU"].isin(["MOTORIN","CNG"])].copy()
print(f"Ariza: {len(df):,}, arac: {df[chr(34)+chr(75)+chr(65)+chr(80)+chr(73)+chr(78)+chr(79)+chr(34)].nunique():,}" if False else f"Ariza: {len(df):,}, arac: {df['KAPINO'].nunique():,}")

# Arac temel profil
arac = df.groupby("KAPINO").agg(
    MARKA=("MARKA","first"),
    MODEL=("MODEL","first"),
    MODELYILI=("MODELYILI","first"),
    ARACCINSI=("ARACCINSI","first"),
    GARAJ=("GARAJ","first"),
    YAKITTURU=("YAKITTURU","first"),
    n_ariza=("ciddi_ariza","count"),
    ort_skor=("ciddiyet_skoru","mean"),
    ciddi_oran=("ciddi_ariza","mean"),
).reset_index()
arac["yas"] = 2025 - arac["MODELYILI"]
print(f"Arac profili: {len(arac):,}")

# === V6 EKLE 1: egim_maruziyet (A3) ===
ah = pd.read_csv("panel_data/temiz_veri/arac_gunluk_hatlar.csv", low_memory=False)
with open(r"panel_data/hat_elevation.json", encoding="utf-8") as f:
    he_raw = _json.load(f)
he = pd.DataFrame([{"HATKODU": k, "rakim": v.get("rakım_farkı", 0), "tirm": v.get("tırmanma_m", 0)}
                   for k, v in he_raw.items()])
def mm_norm(s, q=None):
    if q is not None: s = s.clip(upper=s.quantile(q))
    mn, mx = s.min(), s.max()
    return ((s - mn) / (mx - mn) * 100).round(1) if mx > mn else pd.Series(0.0, index=s.index)
he["norm_r"] = mm_norm(he["rakim"], q=0.99)
he["norm_t"] = mm_norm(he["tirm"], q=0.99)
he["egim_puan"] = (he["norm_r"]*0.4 + he["norm_t"]*0.6).round(1)
ah_m = ah.merge(he[["HATKODU","egim_puan"]], on="HATKODU", how="left")
arac_egim = ah_m.dropna(subset=["egim_puan"]).groupby("KAPINO").apply(
    lambda g: np.average(g["egim_puan"], weights=g["SEFER_SAYISI"].clip(lower=0.01))
).reset_index(name="egim_maruziyet")
arac = arac.merge(arac_egim, on="KAPINO", how="left")
arac["egim_maruziyet"] = arac["egim_maruziyet"].fillna(arac["egim_maruziyet"].median())

# === V6 EKLE 2: son_kaza_gun (A4) - sadece varligini isaretle, gercek hesap A4'te ===
# Burada placeholder olarak 0 atayalim, A4 sonuclari kullanilabilir
arac["son_kaza_gun"] = 365  # placeholder: kaza olmamis varsayilan

# === V6 EKLE 3-6: 4 garaj feature (A5) ===
garaj_arac = df.groupby("GARAJ").agg(
    garaj_ort_skor=("ciddiyet_skoru","mean"),
    garaj_ciddi_oran=("ciddi_ariza","mean"),
).reset_index()
arac = arac.merge(garaj_arac, on="GARAJ", how="left")

# garaj_sistem_lift: arac garajinin "ARIZAUSTKODTANIM" lift'i
genel_ciddi = df["ciddi_ariza"].mean()
gsis = df.groupby(["GARAJ","ARIZAUSTKODTANIM"]).agg(
    n=("ciddi_ariza","count"),
    sis_oran=("ciddi_ariza","mean"),
).reset_index()
gsis["lift"] = gsis["sis_oran"] / genel_ciddi
# Her arac icin garaj × sistem ortalama lift (basitlestirme)
arac_gsl = df.groupby("KAPINO").apply(
    lambda g: gsis[gsis["GARAJ"]==g["GARAJ"].iloc[0]]["lift"].mean()
).reset_index(name="garaj_sistem_lift")
arac = arac.merge(arac_gsl, on="KAPINO", how="left")
arac["garaj_sistem_lift"] = arac["garaj_sistem_lift"].fillna(1.0)

# garaj_marka_lift: garaj × marka çifti için lift
gml = df.groupby(["GARAJ","MARKA"]).agg(
    m_oran=("ciddi_ariza","mean"),
).reset_index()
gml["garaj_marka_lift"] = gml["m_oran"] / genel_ciddi
arac = arac.merge(gml[["GARAJ","MARKA","garaj_marka_lift"]], on=["GARAJ","MARKA"], how="left")
arac["garaj_marka_lift"] = arac["garaj_marka_lift"].fillna(1.0)

# === V6 EKLE 7: yakit_turu_cng (A7) ===
arac["yakit_turu_cng"] = (arac["YAKITTURU"]=="CNG").astype(int)

# === V6 EKLE 8: verimsizlik_skoru (A7) ===
tuketim = pd.DataFrame([
    ("OTOKAR","KENT 290LF",40),("OTOKAR","KENT XL",60),
    ("MERCEDES","CITARO 0530",39),("MERCEDES","CITARO 0530 G",58),
    ("MERCEDES","CONECTO G",62),("MERCEDES","CONECTO",42),
    ("MERCEDES","CAPACITY",65),
    ("BMC","PROCITY TR",41),("BMC","PROCITY",41),
    ("KARSAN","AVANCITY S PLUS",58),("KARSAN","AVANCITY CNG",52),
    ("TEMSA","AVENUE LF CNG",50),
    ("AKIA","ULTRA LF12",40),("AKIA","LF25",60),
], columns=["MARKA","MODEL","tuketim_100km"])
arac = arac.merge(tuketim, on=["MARKA","MODEL"], how="left")
arac["tuketim_100km"] = arac["tuketim_100km"].fillna(arac["tuketim_100km"].median())
def norm_skor(s):
    if s.max() > s.min():
        return ((s - s.min())/(s.max() - s.min()) * 100).round(1)
    return pd.Series(0.0, index=s.index)
yas_norm = norm_skor(arac["yas"])
tuk_norm = norm_skor(arac["tuketim_100km"])
arac["verimsizlik_skoru"] = (yas_norm * tuk_norm / 100).round(2)

# === V6 EKLE 9: hat_zorluk (A9) - her arac icin agirlikli ortalama ===
# Hat zorluk hesabi
hat_ariza = df.groupby("HATKODU").agg(
    hat_ort_skor=("ciddiyet_skoru","mean"),
    hat_uzunluk=("HATUZUNLUK","mean"),
).reset_index()
hat_sefer = ah.groupby("HATKODU").agg(
    sefer_top=("SEFER_SAYISI","sum"),
    n_gun=("TARIH","nunique"),
).reset_index()
hat_sefer["gunluk_sefer"] = hat_sefer["sefer_top"] / hat_sefer["n_gun"].clip(lower=1)
hat = hat_ariza.merge(hat_sefer, on="HATKODU", how="left").merge(he[["HATKODU","egim_puan"]], on="HATKODU", how="left").fillna(0)
hat["n_egim"] = norm_skor(hat["egim_puan"])
hat["n_uzun"] = norm_skor(hat["hat_uzunluk"])
hat["n_ariza"] = norm_skor(hat["hat_ort_skor"])
hat["n_trafik"] = norm_skor(hat["gunluk_sefer"])
hat["hat_zorluk"] = (hat["n_egim"]*0.30 + hat["n_uzun"]*0.20 + hat["n_ariza"]*0.30 + hat["n_trafik"]*0.20).round(1)
ah_h = ah.merge(hat[["HATKODU","hat_zorluk"]], on="HATKODU", how="left")
arac_hz = ah_h.dropna(subset=["hat_zorluk"]).groupby("KAPINO").apply(
    lambda g: np.average(g["hat_zorluk"], weights=g["SEFER_SAYISI"].clip(lower=0.01))
).reset_index(name="hat_zorluk")
arac = arac.merge(arac_hz, on="KAPINO", how="left")
arac["hat_zorluk"] = arac["hat_zorluk"].fillna(arac["hat_zorluk"].median())

# === HEDEF DEGISKEN ===
# ciddi_oran zaten var (continuous), ciddi_ariza_binary (median split)
arac["target_binary"] = (arac["ciddi_oran"] >= arac["ciddi_oran"].median()).astype(int)

print()
print("=== V6 EKLE 9 FEATURE TABLOSU ===")
v6_ekle = ["egim_maruziyet","son_kaza_gun","garaj_sistem_lift","garaj_ort_skor","garaj_ciddi_oran","garaj_marka_lift","yakit_turu_cng","verimsizlik_skoru","hat_zorluk"]
print(arac[["KAPINO"]+v6_ekle].head())
print()
print(arac[v6_ekle].describe().round(3))


Ariza: 58,557, arac: 3,508
Arac profili: 3,508

=== V6 EKLE 9 FEATURE TABLOSU ===
  KAPINO  egim_maruziyet  son_kaza_gun  garaj_sistem_lift  garaj_ort_skor  \
0  A3400       50.548193           365           1.286078        3.713335   
1  A3401       50.997272           365           1.286078        3.713335   
2  A3402       51.830960           365           1.286078        3.713335   
3  A3403       50.835746           365           1.286078        3.713335   
4  A3404       51.007753           365           1.286078        3.713335   

   garaj_ciddi_oran  garaj_marka_lift  yakit_turu_cng  verimsizlik_skoru  \
0          0.403428          1.105166               0               8.97   
1          0.403428          1.105166               0               8.97   
2          0.403428          1.105166               0               8.97   
3          0.403428          1.105166               0               8.97   
4          0.403428          1.105166               0               8.97   

---
## 2. V6 Aday 4 Feature'i Hesapla + Leakage Test

- cascade_risk_skor: 24h ici aynı aracta tekrar arıza orani
- sistem_cas_lift: MASTER_CASCADE'den agrege
- farkli_arac_sayisi: sofor cesitliligi per arac (A2)
- anomali_skor: A9'dan yeniden (hat_zorluk × kritiklik)


In [2]:
# BOLUM 2: V6 aday feature
# === Aday 1: cascade_risk_skor (24h ici tekrar arıza orani) ===
df_sort = df.sort_values(["KAPINO","OLAYTARIHI"]).reset_index(drop=True)
df_sort["arac_prev_time"] = df_sort.groupby("KAPINO")["OLAYTARIHI"].shift(1)
df_sort["sure_gec_saat"] = (df_sort["OLAYTARIHI"] - df_sort["arac_prev_time"]).dt.total_seconds() / 3600
df_sort["cascade_24h"] = (df_sort["sure_gec_saat"] <= 24).astype(int)
cas_arac = df_sort.groupby("KAPINO")["cascade_24h"].agg(["sum","mean"]).reset_index()
cas_arac.columns = ["KAPINO","cascade_event_n","cascade_risk_skor"]
arac = arac.merge(cas_arac, on="KAPINO", how="left").fillna(0)
print(f"cascade_risk_skor: mean={arac['cascade_risk_skor'].mean():.3f}, max={arac['cascade_risk_skor'].max():.3f}")

# === Aday 2: sistem_cas_lift (MASTER_CASCADE'den) ===
master = pd.read_csv("panel_data/temiz_veri/MASTER_CASCADE_TEST_SONUCLARI.csv")
print(f"\nMASTER_CASCADE kayit: {len(master)}")
print(master.head(3))
# Her arac icin: o aracın yasadıgı sistem geçişlerin ortalama lift skoru
# df_sort cascade kayıtları icin "Kaynak_A" → "Hedef_B" eslesmesi
# Basitlestirme: arac başına ortalama lift skoru (genel master ortalaması zaten)
ortalama_lift = master["LIFT_Skoru"].mean()
arac["sistem_cas_lift"] = ortalama_lift  # placeholder - gercek arac başına ayrıntı A1'de
# Daha iyi: cascade_event_n yüksek olanlar daha cok lift maruz kalıyor
arac["sistem_cas_lift"] = arac["cascade_event_n"] * ortalama_lift / arac["cascade_event_n"].mean()
print(f"sistem_cas_lift: mean={arac['sistem_cas_lift'].mean():.3f}")

# === Aday 3: farkli_arac_sayisi (A2 sofor cesitliligi) ===
sof = df.dropna(subset=["SOFOR_SICILNO"]).copy()
sof["SOFOR_SICILNO"] = sof["SOFOR_SICILNO"].astype(str)
arac_sof = sof.groupby("KAPINO")["SOFOR_SICILNO"].nunique().reset_index(name="farkli_sofor_sayisi")
arac = arac.merge(arac_sof, on="KAPINO", how="left").fillna(0)
print(f"\nfarkli_sofor_sayisi: mean={arac['farkli_sofor_sayisi'].mean():.1f}, max={arac['farkli_sofor_sayisi'].max()}")

# === Aday 4: anomali_skor (A9'dan) ===
arac["anomali_skor"] = (arac["hat_zorluk"] * arac["ciddi_oran"] * 100 / 100).round(2)
# Daha dogru: arac_kritiklik × hat_zorluk, ama burada basit proxy
print(f"\nanomali_skor: mean={arac['anomali_skor'].mean():.2f}")

# === LEAKAGE TEST ===
print()
print("="*70)
print("V6 ADAY 4 FEATURE LEAKAGE TESTI")
print("="*70)
# Train/Test split
df_sort_all = df.sort_values("OLAYTARIHI").reset_index(drop=True)
median_t = df_sort_all["OLAYTARIHI"].median()
train_df = df_sort_all[df_sort_all["OLAYTARIHI"] < median_t]
test_df = df_sort_all[df_sort_all["OLAYTARIHI"] >= median_t]
test_arac_target = test_df.groupby("KAPINO")["ciddiyet_skoru"].mean().reset_index(name="test_target")
arac_lt = arac.merge(test_arac_target, on="KAPINO", how="left").dropna(subset=["test_target"])
print(f"\nLeakage test seti (test verisi olan): {len(arac_lt):,} arac")

adaylar = ["cascade_risk_skor","sistem_cas_lift","farkli_sofor_sayisi","anomali_skor"]
print()
print(f"{'Feature':25s} {'Full r':>10s} {'TestOnly r':>12s} {'Dusus %':>10s} {'Karar':>15s}")
print("-"*80)
for f in adaylar:
    full_r = arac[f].corr(arac["ort_skor"])
    test_r = arac_lt[f].corr(arac_lt["test_target"])
    dusus = (1 - abs(test_r)/abs(full_r))*100 if abs(full_r) > 0.01 else 0
    if dusus > 50:
        karar = "ATLA (leakage)"
    elif dusus > 30:
        karar = "RISKLI"
    else:
        karar = "GUVENLI"
    print(f"{f:25s} {full_r:>+10.4f} {test_r:>+12.4f} {dusus:>9.1f}% {karar:>15s}")


cascade_risk_skor: mean=0.119, max=0.500

MASTER_CASCADE kayit: 557
                Kaynak_A                             Hedef_B  \
0                 Destek             YAKIT POMPASI ARIZALARI   
1           ROT AYARLARI              DİFERANSİYEL ARIZALARI   
2  KWS SİSTEMİ ARIZALARI  İLAVE DİREKSİYON SİSTEMİ ARIZALARI   

   Gözlemlenen_Adet  LIFT_Skoru  P_Degeri  Medyan_Sure_Saat  Arac_Sayisi  \
0                 3    9.814358  0.000063         34.760000            3   
1                 3    7.151225  0.001278          9.896389            3   
2                 6    6.436587  0.000002         54.804722            6   

  Güven_Seviyesi  
0         Yüksek  
1         Yüksek  
2         Yüksek  
sistem_cas_lift: mean=1.211

farkli_sofor_sayisi: mean=14.2, max=57

anomali_skor: mean=12.17

V6 ADAY 4 FEATURE LEAKAGE TESTI

Leakage test seti (test verisi olan): 3,439 arac

Feature                       Full r   TestOnly r    Dusus %           Karar
---------------------------------------

---
## 3. Pairwise Correlation Matrix (Multicollinearity Ilk Gozlem)

Tum V6 EKLE + gecen V6 aday feature'lari icin correlation matrix.


In [3]:
# BOLUM 3: Correlation matrix
import matplotlib.pyplot as plt

tum_features = ["egim_maruziyet","garaj_sistem_lift","garaj_ort_skor","garaj_ciddi_oran","garaj_marka_lift",
                "yakit_turu_cng","verimsizlik_skoru","hat_zorluk",
                "cascade_risk_skor","sistem_cas_lift","farkli_sofor_sayisi","anomali_skor",
                "yas"]

corr = arac[tum_features].corr()
print("=== PEARSON CORRELATION MATRIX ===")
print(corr.round(2).to_string())

# Yuksek korelasyon (>0.7) ciftleri
print()
print("=== YUKSEK KORELASYON (|r| > 0.7) ===")
high_corr = []
for i in range(len(tum_features)):
    for j in range(i+1, len(tum_features)):
        f1, f2 = tum_features[i], tum_features[j]
        r = corr.loc[f1, f2]
        if abs(r) > 0.7:
            high_corr.append((f1, f2, r))
            print(f"  {f1} <-> {f2}: r={r:+.3f}")
if not high_corr:
    print("  Yok (tum cifler |r| < 0.7)")

# Orta korelasyon (0.5-0.7)
print()
print("=== ORTA KORELASYON (0.5 <= |r| < 0.7) ===")
for i in range(len(tum_features)):
    for j in range(i+1, len(tum_features)):
        f1, f2 = tum_features[i], tum_features[j]
        r = corr.loc[f1, f2]
        if 0.5 <= abs(r) < 0.7:
            print(f"  {f1} <-> {f2}: r={r:+.3f}")


=== PEARSON CORRELATION MATRIX ===
                     egim_maruziyet  garaj_sistem_lift  garaj_ort_skor  garaj_ciddi_oran  garaj_marka_lift  yakit_turu_cng  verimsizlik_skoru  hat_zorluk  cascade_risk_skor  sistem_cas_lift  farkli_sofor_sayisi  anomali_skor   yas
egim_maruziyet                 1.00               0.53            0.29              0.48              0.48           -0.46               0.19        0.57               0.13             0.16                 0.23          0.40  0.31
garaj_sistem_lift              0.53               1.00            0.54              0.86              0.73           -0.33               0.25        0.60               0.27             0.35                 0.48          0.49  0.25
garaj_ort_skor                 0.29               0.54            1.00              0.85              0.72            0.18               0.27        0.42               0.13             0.16                 0.23          0.38  0.48
garaj_ciddi_oran               0.48      

---
## 4. VIF (Variance Inflation Factor) Multicollinearity Testi

VIF > 10 = ciddi multicollinearity. VIF > 5 = orta. < 5 = güvenli.


In [4]:
# BOLUM 4: VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X = arac[tum_features].copy()
# NaN doldur
for c in X.columns:
    X[c] = X[c].fillna(X[c].median())
X_c = add_constant(X)

vif_data = pd.DataFrame()
vif_data["Feature"] = X_c.columns
vif_data["VIF"] = [variance_inflation_factor(X_c.values, i) for i in range(X_c.shape[1])]
vif_data = vif_data[vif_data["Feature"]!="const"].sort_values("VIF", ascending=False).reset_index(drop=True)

print("=== VIF (Variance Inflation Factor) ===")
print(vif_data.to_string(index=False))

# Karar
print()
print("=== KARAR ===")
yuksek = vif_data[vif_data["VIF"] > 10]
orta = vif_data[(vif_data["VIF"] > 5) & (vif_data["VIF"] <= 10)]
guvenli = vif_data[vif_data["VIF"] <= 5]
print(f"VIF > 10 (CIDDI multicollinearity): {len(yuksek)} feature")
if len(yuksek) > 0:
    print(f"  Atilmasi onerilen: {yuksek['Feature'].tolist()}")
print(f"VIF 5-10 (ORTA): {len(orta)} feature")
print(f"VIF < 5 (GUVENLI): {len(guvenli)} feature")
print(f"  Guvenli: {guvenli['Feature'].tolist()}")


=== VIF (Variance Inflation Factor) ===
            Feature       VIF
   garaj_ciddi_oran 31.966233
     garaj_ort_skor 16.287358
  garaj_sistem_lift 10.444598
    sistem_cas_lift  8.151669
farkli_sofor_sayisi  5.169839
         hat_zorluk  4.934600
   garaj_marka_lift  4.828812
  cascade_risk_skor  3.911926
                yas  3.203129
  verimsizlik_skoru  2.756067
     yakit_turu_cng  2.533664
     egim_maruziyet  2.515753
       anomali_skor  1.966825

=== KARAR ===
VIF > 10 (CIDDI multicollinearity): 3 feature
  Atilmasi onerilen: ['garaj_ciddi_oran', 'garaj_ort_skor', 'garaj_sistem_lift']
VIF 5-10 (ORTA): 2 feature
VIF < 5 (GUVENLI): 8 feature
  Guvenli: ['hat_zorluk', 'garaj_marka_lift', 'cascade_risk_skor', 'yas', 'verimsizlik_skoru', 'yakit_turu_cng', 'egim_maruziyet', 'anomali_skor']


---
## 5. Master Feature CSV Export

ML V6 icin reproducible pipeline cikti.


In [5]:
# BOLUM 5: Master CSV
# Duplicate yas problemi: master_cols + tum_features arasinda yas tekrar ediyor
meta_cols = ["KAPINO","GARAJ","MARKA","MODEL","YAKITTURU","ARACCINSI",
             "n_ariza","ort_skor","ciddi_oran","target_binary"]
# yas zaten tum_features icinde
master_cols = meta_cols + tum_features
master_cols = list(dict.fromkeys(master_cols))  # duplicate temizle, sira koru

master = arac[master_cols].copy()
master.to_csv("features_final_master.csv", index=False)
print(f"Master CSV export: features_final_master.csv ({len(master):,} satir, {len(master.columns)} kolon)")
print()
print("Kolonlar:")
for c in master.columns:
    nans = master[c].isna().sum()
    dt = str(master[c].dtype)
    print(f"  {c:25s} - dtype: {dt:10s} - NaN: {nans}")


Master CSV export: features_final_master.csv (3,508 satir, 23 kolon)

Kolonlar:
  KAPINO                    - dtype: object     - NaN: 0
  GARAJ                     - dtype: object     - NaN: 0
  MARKA                     - dtype: object     - NaN: 0
  MODEL                     - dtype: object     - NaN: 0
  YAKITTURU                 - dtype: object     - NaN: 0
  ARACCINSI                 - dtype: object     - NaN: 0
  n_ariza                   - dtype: int64      - NaN: 0
  ort_skor                  - dtype: float64    - NaN: 0
  ciddi_oran                - dtype: float64    - NaN: 0
  target_binary             - dtype: int64      - NaN: 0
  egim_maruziyet            - dtype: float64    - NaN: 0
  garaj_sistem_lift         - dtype: float64    - NaN: 0
  garaj_ort_skor            - dtype: float64    - NaN: 0
  garaj_ciddi_oran          - dtype: float64    - NaN: 0
  garaj_marka_lift          - dtype: float64    - NaN: 0
  yakit_turu_cng            - dtype: int64      - NaN: 0
  verims

---
## 6. Final Karar Tablosu

Multicollinearity + leakage sonuçlarına göre ML V6 için final feature listesi.


In [6]:
# BOLUM 6: Final karar
print("="*70)
print("FEATURES_FINAL — ML V6 ICIN FINAL KARAR")
print("="*70)

karar_tablosu = []
for f in tum_features:
    if f == "yas":
        # yas zaten temel feature
        karar_tablosu.append({"Feature": f, "Kaynak": "Temel", "r_ort_skor": arac[f].corr(arac["ort_skor"]), "VIF": vif_data[vif_data["Feature"]==f]["VIF"].iloc[0] if (vif_data["Feature"]==f).any() else None, "Karar":"YAS - kullan"})
    elif f in ["cascade_risk_skor","sistem_cas_lift","farkli_sofor_sayisi","anomali_skor"]:
        # Aday — leakage testinden geçti mi?
        full_r = arac[f].corr(arac["ort_skor"])
        test_r = arac_lt[f].corr(arac_lt["test_target"])
        dusus = (1 - abs(test_r)/abs(full_r))*100 if abs(full_r) > 0.01 else 0
        v = vif_data[vif_data["Feature"]==f]["VIF"].iloc[0] if (vif_data["Feature"]==f).any() else None
        if dusus > 50:
            karar = "ATLA"
        elif v > 10:
            karar = "VIF YUKSEK - test"
        else:
            karar = "V6 EKLE (aday gecti)"
        karar_tablosu.append({"Feature": f, "Kaynak": "Aday", "r_ort_skor": full_r, "VIF": v, "Karar": karar})
    else:
        # V6 EKLE feature
        full_r = arac[f].corr(arac["ort_skor"])
        v = vif_data[vif_data["Feature"]==f]["VIF"].iloc[0] if (vif_data["Feature"]==f).any() else None
        if v > 10:
            karar = "VIF YUKSEK - dusunuluyor"
        else:
            karar = "V6 EKLE"
        karar_tablosu.append({"Feature": f, "Kaynak": "V6 EKLE", "r_ort_skor": full_r, "VIF": v, "Karar": karar})

karar_df = pd.DataFrame(karar_tablosu).round(3)
print(karar_df.to_string(index=False))

print()
print("=== OZET ===")
final_ekle = karar_df[karar_df["Karar"].str.contains("EKLE")]
final_atla = karar_df[karar_df["Karar"]=="ATLA"]
final_vif = karar_df[karar_df["Karar"].str.contains("VIF")]
print(f"FINAL EKLE: {len(final_ekle)} feature")
print(f"  Liste: {final_ekle['Feature'].tolist()}")
print(f"VIF problemli (test gerekli): {len(final_vif)}")
if len(final_vif) > 0:
    print(f"  Liste: {final_vif['Feature'].tolist()}")
print(f"ATLA: {len(final_atla)}")


FEATURES_FINAL — ML V6 ICIN FINAL KARAR
            Feature  Kaynak  r_ort_skor    VIF                    Karar
     egim_maruziyet V6 EKLE       0.166  2.516                  V6 EKLE
  garaj_sistem_lift V6 EKLE       0.278 10.445 VIF YUKSEK - dusunuluyor
     garaj_ort_skor V6 EKLE       0.488 16.287 VIF YUKSEK - dusunuluyor
   garaj_ciddi_oran V6 EKLE       0.424 31.966 VIF YUKSEK - dusunuluyor
   garaj_marka_lift V6 EKLE       0.417  4.829                  V6 EKLE
     yakit_turu_cng V6 EKLE       0.035  2.534                  V6 EKLE
  verimsizlik_skoru V6 EKLE       0.146  2.756                  V6 EKLE
         hat_zorluk V6 EKLE       0.237  4.935                  V6 EKLE
  cascade_risk_skor    Aday       0.035  3.912     V6 EKLE (aday gecti)
    sistem_cas_lift    Aday       0.064  8.152     V6 EKLE (aday gecti)
farkli_sofor_sayisi    Aday       0.119  5.170     V6 EKLE (aday gecti)
       anomali_skor    Aday       0.721  1.967     V6 EKLE (aday gecti)
                yas   Te

---
## 7. Tree Model Değerlendirmesi — Final Karar Revize

Bölüm 4'te VIF testi yaptık (linear model varsayımı). Ancak ML V6 için **tree-based model (XGBoost/LightGBM)** kullanılacak. Tree modeller multicollinearity'ye **dayanıklıdır:**
- Her node'da en iyi split seçilir
- Korele feature'lar redundant olur (accuracy bozulmaz)
- Sadece feature importance bölünür (yorumlanabilirlik kaybı)

### Bu Bilgi Işığında Yeniden Değerlendirme:

| Feature | VIF | Linear Karar | Tree Model Karar |
|---|---|---|---|
| garaj_sistem_lift | 10.44 | ATLA | **KEEP** (farklı bilgi: genel sistem kalitesi) |
| garaj_ort_skor | 16.29 | ATLA | ATLA (garaj_marka_lift ile r=0.72, dublike) |
| garaj_ciddi_oran | 31.97 | ATLA | ATLA (garaj_sistem_lift ile r=0.86, redundant) |
| cascade_risk_skor | 3.91 (OK) | RISKLI | **KEEP** (r zayıf ama SHAP testi için) |

### Modern ML Disiplini: "Include-Then-Prune"
1. V6.0: Geniş feature seti ile başla (9 feature)
2. SHAP/feature_importance ile elek
3. V6.1: Zayıf feature'ları çıkar

### Final 9 Feature Listesi (Strategic):
- yas (Temel)
- egim_maruziyet (A3 Topografya)
- garaj_sistem_lift (A5, genel garaj kalitesi)
- garaj_marka_lift (A5, marka-garaj uyumu — farklı bilgi)
- yakit_turu_cng (A7 Yakıt)
- verimsizlik_skoru (A7 Yakıt)
- hat_zorluk (A9 Atama)
- cascade_risk_skor (A1, SHAP testi adayı)
- farkli_sofor_sayisi (A2 Şoför)


In [7]:
# BOLUM 7: Tree Model — 9 Feature Final
v6_final = [
    "yas",
    "egim_maruziyet",
    "garaj_sistem_lift",  # VIF 10.4 — tree-tolerable
    "garaj_marka_lift",
    "yakit_turu_cng",
    "verimsizlik_skoru",
    "hat_zorluk",
    "cascade_risk_skor",  # zayif r — SHAP elek adayi
    "farkli_sofor_sayisi",
]
atilan = [
    ("garaj_ort_skor", "VIF 16.29 + r=0.72 garaj_marka_lift ile (redundant)"),
    ("garaj_ciddi_oran", "VIF 31.97 + r=0.86 garaj_sistem_lift ile (dublike)"),
    ("sistem_cas_lift", "r=0.82 cascade_risk_skor ile (dublike)"),
    ("anomali_skor", "Target leakage (hat_zorluk × ciddi_oran formulunde target var)"),
    ("son_kaza_gun", "Validation'da placeholder = 365. ML V6'da gercek hesap gerek"),
    ("sofor_glob_skor", "A2'de %81 leakage tespit edildi"),
    ("gunluk_sefer_sayisi", "A6 yorgunluk: %93 leakage + ters yon"),
    ("son_30g_top", "A6 yorgunluk: sinyal yok (r=-0.014)"),
    ("kritiklik_skoru", "A8: stability r=0.15 zayif (operasyonel only)"),
]

print("=" * 70)
print("V6 FINAL FEATURE LISTESI (9 feature)")
print("=" * 70)
for i, f in enumerate(v6_final, 1):
    full_r = arac[f].corr(arac["ort_skor"])
    vif_val = vif_data[vif_data["Feature"]==f]["VIF"].iloc[0] if (vif_data["Feature"]==f).any() else None
    not_str = ""
    if f == "cascade_risk_skor":
        not_str = " (zayif r — SHAP eleg adayi)"
    elif f == "garaj_sistem_lift":
        not_str = " (VIF 10.4 — tree-tolerable)"
    print(f"{i}. {f:25s}  r={full_r:+.3f}, VIF={vif_val:.2f}{not_str}")

print()
print("=" * 70)
print("ATILAN FEATURES (9)")
print("=" * 70)
for f, sebep in atilan:
    print(f"  {f:25s}  — {sebep}")

print()
print("=" * 70)
print("V6 STRATEJI: INCLUDE-THEN-PRUNE")
print("=" * 70)
print("1. V6.0: 9 feature ile XGBoost/LightGBM fit")
print("2. SHAP analizi: hangi feature gercekten katki sagliyor?")
print("3. V6.1: feature_importance < %1 olanlari cikar")
print("4. V5 (AUC=0.762) vs V6 karsilastirma")
print()
print("=== MASTER CSV V2 EXPORT (9 feature) ===")
v6_cols = ["KAPINO","GARAJ","MARKA","MODEL","YAKITTURU","ARACCINSI",
           "n_ariza","ort_skor","ciddi_oran","target_binary"] + v6_final
master_v2 = arac[list(dict.fromkeys(v6_cols))].copy()
master_v2.to_csv("features_final_v2_9feature.csv", index=False)
print(f"Yeni CSV: features_final_v2_9feature.csv ({len(master_v2):,} satir, {len(master_v2.columns)} kolon)")


V6 FINAL FEATURE LISTESI (9 feature)
1. yas                        r=+0.209, VIF=3.20
2. egim_maruziyet             r=+0.166, VIF=2.52
3. garaj_sistem_lift          r=+0.278, VIF=10.44 (VIF 10.4 — tree-tolerable)
4. garaj_marka_lift           r=+0.417, VIF=4.83
5. yakit_turu_cng             r=+0.035, VIF=2.53
6. verimsizlik_skoru          r=+0.146, VIF=2.76
7. hat_zorluk                 r=+0.237, VIF=4.93
8. cascade_risk_skor          r=+0.035, VIF=3.91 (zayif r — SHAP eleg adayi)
9. farkli_sofor_sayisi        r=+0.119, VIF=5.17

ATILAN FEATURES (9)
  garaj_ort_skor             — VIF 16.29 + r=0.72 garaj_marka_lift ile (redundant)
  garaj_ciddi_oran           — VIF 31.97 + r=0.86 garaj_sistem_lift ile (dublike)
  sistem_cas_lift            — r=0.82 cascade_risk_skor ile (dublike)
  anomali_skor               — Target leakage (hat_zorluk × ciddi_oran formulunde target var)
  son_kaza_gun               — Validation'da placeholder = 365. ML V6'da gercek hesap gerek
  sofor_glob_skor      